In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# 1. Load the raw data paths
print("Loading data partitions...")
fraud_data = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_data = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

# --- Re-applying the engineered features from your EDA phase ---
fraud_data = fraud_data.drop_duplicates()
fraud_data['signup_time'] = pd.to_datetime(fraud_data['signup_time'])
fraud_data['purchase_time'] = pd.to_datetime(fraud_data['purchase_time'])

# Direct range mapping logic 
fraud_data['ip_int'] = fraud_data['ip_address'].fillna(0).astype(np.int64)
ip_df_sorted = ip_data.sort_values(by='lower_bound_ip_address').reset_index(drop=True)
idx = np.searchsorted(ip_df_sorted['lower_bound_ip_address'].values, fraud_data['ip_int'].values) - 1
idx = np.clip(idx, 0, len(ip_df_sorted) - 1)
valid_mask = (fraud_data['ip_int'].values >= ip_df_sorted['lower_bound_ip_address'].values[idx]) & \
             (fraud_data['ip_int'].values <= ip_df_sorted['upper_bound_ip_address'].values[idx])
fraud_data['country'] = np.where(valid_mask, ip_df_sorted['country'].values[idx], 'Unknown')

# Deriving transaction velocity fields
fraud_data['time_since_signup'] = (fraud_data['purchase_time'] - fraud_data['signup_time']).dt.total_seconds()
fraud_data['hour_of_day'] = fraud_data['purchase_time'].dt.hour
fraud_data['day_of_week'] = fraud_data['purchase_time'].dt.dayofweek
fraud_data['user_device_count'] = fraud_data.groupby('device_id')['user_id'].transform('count')
fraud_data['user_ip_count'] = fraud_data.groupby('ip_address')['user_id'].transform('count')

# 2. Categorical Variable One-Hot Encoding
categorical_features = ['source', 'browser', 'sex', 'country']
# drop_first=True prevents the dummy variable trap (multicollinearity)
encoded_df = pd.get_dummies(fraud_data, columns=categorical_features, drop_first=True)

# Isolate predictive arrays from string markers and timestamps
drop_identifiers = ['user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address', 'ip_int', 'class']
X = encoded_df.drop(columns=drop_identifiers)
y = encoded_df['class']

# 3. Stratified Train-Test Split (80% Train, 20% Test)
# Justification: Stratification keeps the exact fraud-to-legitimate ratio identical in both splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Feature Scaling (Standard Normalization)
scaler = StandardScaler()
numerical_cols = ['purchase_value', 'time_since_signup', 'hour_of_day', 'day_of_week', 'user_device_count', 'user_ip_count']
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# 5. Overcome Class Imbalance using SMOTE strictly on Training Set
print("\n--- Distribution BEFORE Resampling ---")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\n--- Distribution AFTER SMOTE Resampling ---")
print(y_train_resampled.value_counts())

Loading data partitions...

--- Distribution BEFORE Resampling ---
class
0    109568
1     11321
Name: count, dtype: int64

--- Distribution AFTER SMOTE Resampling ---
class
0    109568
1    109568
Name: count, dtype: int64
